In [ ]:
import html
from pathlib import Path
from collections import defaultdict
import json 

full_path = '/Users/maremphd/VsCode ProjectFolder/GBV-CounterSpeech/data/card_creation_comparison/runs/confronto-02/reranking'

ROOT = Path(full_path)
paths = sorted(ROOT.glob('*.jsonl'))
if not paths:
    raise ValueError(f'No JSONL files found in {ROOT}')


In [33]:
top_n=3

settings = {}
required = {'label', 'definition', 'text', 'chunk_id', 'reranker_rank'}
for path in paths:
    labels = defaultdict(list)
    with path.open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            row = json.loads(line)
            missing = required - row.keys()
            if missing:
                raise ValueError(f'{path}:{line_number}: missing {sorted(missing)}')
            if row.get('configuration_id', path.stem) != path.stem:
                raise ValueError(f'{path}:{line_number}: configuration_id differs from filename')
            if len(labels[row['label']]) < top_n:
                labels[row['label']].append(row)
    if not labels:
        raise ValueError(f'No entries found in {path}')        
    settings[path.stem] = (path, dict(labels))

all_metadata = set.union(*(set(metadata) for _, metadata in settings.values()))
for configuration, (_, metadata) in settings.items():
    for info in sorted(all_metadata):
        rows = metadata.get(info, [])
        if len(rows) != top_n:
            raise ValueError(f'{configuration}: expected {top_n} items for {info}, found {len(rows)}')
        if len({row['chunk_id'] for row in rows}) != len(rows):
            raise ValueError(f'{configuration}: duplicate chunk_id among selected items for {info}')

In [40]:
from collections import defaultdict

chunk_counts = defaultdict(int)

for configuration, (_, metadata) in settings.items():
    for label, rows in metadata.items():
        for row in rows:
            key = (label, row['chunk_id'])
            chunk_counts[key] += 1

for (label, chunk_id), count in sorted(chunk_counts.items()):
    print(f'Label: {label}, Chunk: {chunk_id}, Configurations: {count}')

Label: 1.1 threats of harm, Chunk: chunk_00044e8dca3c44a18dde8020, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_1713bcff2114a70bef1ace47, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_1a9fc296cf26ccd722debde3, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_1b382b40c0a3399f4018cd28, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_2dd53f36e40e471ed8a452b1, Configurations: 2
Label: 1.1 threats of harm, Chunk: chunk_4af584fa62b4a5fd2ada61f3, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_74cb83e2de5fc8be38dfc084, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_80f9ab69192e1ba5389d6819, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_8524235361e45ec29078d0fa, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_b9ee7ca290e2a3e97639c489, Configurations: 1
Label: 1.1 threats of harm, Chunk: chunk_df2288a86f3f8a0a56e4e0aa, Configurations: 1
Label: 1.2 incitement and encouragement of harm, Chunk: chunk_212

In [41]:
print("\nChunks selected in all 4 configurations:")

for (label, chunk_id), count in sorted(chunk_counts.items()):
    if count == 4:
        print(f'Label: {label}, Chunk: {chunk_id}')


Chunks selected in all 4 configurations:
Label: 1.2 incitement and encouragement of harm, Chunk: chunk_bee41b5250c2db19ccdee397
Label: 2.2 aggressive and emotive attacks, Chunk: chunk_1713bcff2114a70bef1ace47
Label: 2.2 aggressive and emotive attacks, Chunk: chunk_8eff3ec4f0b38dd6e80f2a48
Label: 4.1 supporting mistreatment of individual women, Chunk: chunk_0822266b42ae7e39859d88f5


In [39]:
for configuration, (_, metadata) in settings.items():
    print(f'\nConfiguration: {configuration}')
    for info, rows in metadata.items():
        print(f'  Info: {info}')
        for row in rows:
            print(f'    Chunk ID: {row["chunk_id"]}, Rank: {row["reranker_rank"]}, Label: {row["label"]}')


Configuration: bge-m3__bge-reranker-v2-m3
  Info: 1.1 threats of harm
    Chunk ID: chunk_df2288a86f3f8a0a56e4e0aa, Rank: 1, Label: 1.1 threats of harm
    Chunk ID: chunk_b9ee7ca290e2a3e97639c489, Rank: 2, Label: 1.1 threats of harm
    Chunk ID: chunk_8524235361e45ec29078d0fa, Rank: 3, Label: 1.1 threats of harm
  Info: 1.2 incitement and encouragement of harm
    Chunk ID: chunk_bee41b5250c2db19ccdee397, Rank: 1, Label: 1.2 incitement and encouragement of harm
    Chunk ID: chunk_9736e15ea4d8d2bba844c7f3, Rank: 2, Label: 1.2 incitement and encouragement of harm
    Chunk ID: chunk_2127cecc05d1ff10c4edfe90, Rank: 3, Label: 1.2 incitement and encouragement of harm
  Info: 2.1 descriptive attacks
    Chunk ID: chunk_8524235361e45ec29078d0fa, Rank: 1, Label: 2.1 descriptive attacks
    Chunk ID: chunk_a271bbf8c38b10d6edfe717e, Rank: 2, Label: 2.1 descriptive attacks
    Chunk ID: chunk_b38bb9b365869b5e28b16150, Rank: 3, Label: 2.1 descriptive attacks
  Info: 2.2 aggressive and emotive 